In [ ]:
import pandas as pd
import numpy as np
# import os
# os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=true --xla_cpu_multi_thread_eigen_num_threads=4"
import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp
from jax import lax

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median


df = pd.read_csv('df.csv')
df

In [ ]:
actuals = df["sales"].values
y = np.log(actuals)

In [ ]:
def make_fourier_series(t, period, order):
    """
    Args
    ----
    t: array-like, time points at which to evaluate the Fourier series
    period: int, the period of the seasonality
    order: int, the number of Fourier terms to include
    """
    sin_terms = jnp.array([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    cos_terms = jnp.array([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
    return jnp.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

x = make_fourier_series(jnp.arange(len(y)), period=365.25, order=3)
x.shape

In [ ]:
def dlt_transition_step(carry, inputs):
    """Damped Local Trend (DLT) transition function
    Args
    ----
    carry: tuple containing the previous level and bias
    inputs: tuple containing the current observation, growth, trend, and seasonality
    """
    # make this partial inputs later
    lev_sm = 0.005
    slp_sm = 0.01
    # damped factor
    theta = 0.9

    lev_prev, slp_prev = carry
    y_t = inputs

    # forecast
    yhat_t = lev_prev + theta * slp_prev

    # update
    new_lev = lev_sm * y_t + (1 - lev_sm) * (lev_prev + theta * slp_prev)
    new_slp = slp_sm * (new_lev - lev_prev) + (1 - slp_sm) * slp_prev

    return (new_lev, new_slp), (new_lev, new_slp, yhat_t)

_, res = lax.scan(dlt_transition_step, (y[0], 0), y)
levs, slps, y_hats = res

In [ ]:
def dlt_model():
    # variance coefficient
    sigma = numpyro.sample("sigma", dist.HalfNormal(0.5))
    beta = numpyro.sample("beta", dist.Normal(0, 0.3).expand([x.shape[1]]))

    # (n_steps, )
    seas = jnp.sum(x * beta, axis=-1)
    
    _, res = lax.scan(dlt_transition_step, (y[0] - seas[0], 0), y - seas)

    _, _, dlt_mu = res
    mu = dlt_mu + seas

    numpyro.deterministic("mu", mu)

    # likelihood
    numpyro.sample("observations", dist.Normal(loc=mu, scale=sigma), obs=y)


init_strategy = init_to_median(num_samples=10)
kernel = NUTS(dlt_model, init_strategy=init_strategy)
mcmc = MCMC(kernel, num_warmup=100, num_samples=100, num_chains=4)
mcmc.run(jax.random.PRNGKey(0))
posteriors_dict = mcmc.get_samples()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
sigma_samples = np.array(posteriors_dict["sigma"])

# generate error
eps_samples = np.transpose(
    np.random.normal(loc=0.0, scale=sigma_samples, size=(len(y), sigma_samples.shape[0])),
    axes=(1, 0)
)
print(eps_samples.shape)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples), q=[0.1, 0.5, 0.9], axis=0)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(y)), actuals, label='Observed', alpha=0.5, color="orange")
ax.plot(np.arange(len(y)), yhat_mid, label='Forecast', linestyle='--', color="dodgerblue")
ax.fill_between(np.arange(len(y)), yhat_lower, yhat_upper, alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('Damped Local Trend Forecast')